# Organização de Imagens por Desempenho do Modelo e Coerência do Grad-CAM
 
Autor:  Matheus de Farias Cavalcanti Santos
 
Atualizado: 06/03/2026
 
## Objetivos

Este notebook tem como objetivo organizar automaticamente as imagens do experimento em uma estrutura de pastas que facilite a análise dos resultados do modelo de classificação e da coerência do Grad-CAM.

A partir da planilha de controle (colunas **LABEL** e **NOME DA IMAGEM**), o notebook localiza cada imagem nas pastas de origem (**grad_negativo_parte1** e **grad_positivo_parte1**) e copia os arquivos para pastas de destino separadas por classe (**POSITIVO** e **NEGATIVO**). Dentro de cada classe, as imagens são agrupadas conforme as combinações entre:

- **MODELO ACERTOU? SIM(1) NÃO(0)**  
- **GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?**

Gerando quatro grupos por classe:
- modelo acerta (1) e gradcam erra (0)
- modelo acerta (1) e gradcam acerta (1)
- modelo erra (0) e gradcam acerta (1)
- modelo erra (0) e gradcam erra (0)

Além disso, o notebook cria uma pasta específica de **REANOTAÇÃO**, contendo subpastas **POSITIVO** e **NEGATIVO**, para armazenar todas as imagens marcadas para reanotação na coluna **REANOTAR? SIM(1) NÃO(0)** com valor igual a **1**.

Dessa forma, a organização gerada permite inspecionar rapidamente:
1) onde o modelo acerta ou erra,  
2) quando o Grad-CAM é coerente ou incoerente,  
3) e quais amostras precisam ser revisadas (reanotação), tornando a análise qualitativa e a auditoria dos resultados muito mais eficientes.

🔗[Link para a planilha de análise das imagens classificação gradcam](https://petrobrasbr.sharepoint.com/:x:/r/teams/IASubmarina/Shared%20Documents/Evid%C3%AAncias-OKRs/1%20-%20Arquivos%20de%20evid%C3%AAncias/2026/CICLO%201/OKR4/An%C3%A1lise%20das%20imagens%20classifica%C3%A7%C3%A3o%20gradcam.xlsx?d=wd5a7ebe833e6498f8fec726b48bc7141&csf=1&web=1&e=f9TFtc)

🔗[Link para as imagens geradas resultantes da execução desse notebook](https://petrobrasbr.sharepoint.com/:f:/r/teams/IASubmarina/Shared%20Documents/Evid%C3%AAncias-OKRs/1%20-%20Arquivos%20de%20evid%C3%AAncias/2026/CICLO%201/OKR4/Imagens%20da%20combina%C3%A7%C3%A3o%20do%20modelo%20com%20gradcam?csf=1&web=1&e=Ah6zBu)

🔗[Link das imagens para reanotação](https://petrobrasbr.sharepoint.com/:f:/r/teams/IASubmarina/Shared%20Documents/Evid%C3%AAncias-OKRs/1%20-%20Arquivos%20de%20evid%C3%AAncias/2026/CICLO%201/OKR4/Imagens%20para%20reanota%C3%A7%C3%A3o?csf=1&web=1&e=2zCGUX)


# Importações necessárias

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import pandas as pd
import shutil
import unicodedata

# Declaração de constantes

In [ ]:
ARQUIVO_EXCEL = Path("[Diretório da planilha]")

PASTA_ORIGEM_NEG = Path(r"[Diretório]")
PASTA_ORIGEM_POS = Path(r"[Diretório]")

# Pastas específicas para copiar imagens de REANOTAÇÃO
PASTA_REANOTACAO_ORIGEM_NEG = Path(r"[Diretório]")
PASTA_REANOTACAO_ORIGEM_POS = Path(r"[Diretório]")

PASTA_SAIDA = Path("[Diretório]")

COL_LABEL = "LABEL"
COL_NOME_IMG = "NOME DA IMAGEM"
COL_MODELO_ACERTOU = "MODELO ACERTOU? SIM(1) NÃO(0)"
COL_GRADCAM_OK = "GRADCAM: ATENÇÃO NO LOCAL CERTO(1) OU ERRADO(0)?"
COL_REANOTAR = "REANOTAR? SIM(1) NÃO(0)"

# Funções necessárias

In [ ]:
def normalizar_texto(txt: str) -> str:
    """
    Normaliza texto removendo espaços extras e padronizando para caixa alta.
    Também remove acentos para evitar inconsistências, caso existam.

    Parâmetros
    ----------
    txt : str
        Texto de entrada.

    Retorno
    -------
    str
        Texto normalizado (sem acento, caixa alta, sem espaços nas pontas).
    """
    if pd.isna(txt):
        return ""
    txt = str(txt).strip().upper()
    txt = unicodedata.normalize("NFKD", txt)
    txt = "".join(ch for ch in txt if not unicodedata.combining(ch))
    return txt


def padronizar_binario(valor) -> Optional[int]:
    """
    Converte valores variados para binário (0 ou 1).

    Aceita:
    - 0/1 numérico
    - '0'/'1'
    - True/False

    Retorno
    -------
    Optional[int]
        0 ou 1 quando conversão possível; None caso inválido/ausente.
    """
    if pd.isna(valor):
        return None

    if isinstance(valor, bool):
        return int(valor)

    s = str(valor).strip()
    if s in {"0", "0.0"}:
        return 0
    if s in {"1", "1.0"}:
        return 1

    try:
        v = int(float(s))
        if v in (0, 1):
            return v
    except Exception:
        pass

    return None


def garantir_pasta(p: Path) -> None:
    """
    Cria a pasta (e pais) caso não exista.

    Parâmetros
    ----------
    p : Path
        Caminho da pasta.
    """
    p.mkdir(parents=True, exist_ok=True)


def validar_arquivos_e_pastas(*paths: Path) -> None:
    """
    Valida existência de arquivos/pastas essenciais.

    Lança exceção se algo não existir.
    """
    faltando = [str(p) for p in paths if not p.exists()]
    if faltando:
        raise FileNotFoundError(f"Os seguintes caminhos não foram encontrados: {faltando}")


def localizar_imagem(nome_imagem: str, pastas_origem: List[Path]) -> Optional[Path]:
    """
    Procura um arquivo de imagem em uma lista de pastas de origem (não recursivo).

    Parâmetros
    ----------
    nome_imagem : str
        Nome do arquivo (com extensão), ex: 'img_001.png'
    pastas_origem : List[Path]
        Pastas onde procurar.

    Retorno
    -------
    Optional[Path]
        Caminho completo para o arquivo encontrado, ou None se não encontrar.
    """
    for pasta in pastas_origem:
        candidato = pasta / nome_imagem
        if candidato.exists() and candidato.is_file():
            return candidato
    return None


def copiar_arquivo(origem: Path, destino: Path) -> None:
    """
    Copia um arquivo preservando metadados.

    Parâmetros
    ----------
    origem : Path
        Caminho do arquivo de origem.
    destino : Path
        Caminho final do arquivo de destino (inclui nome do arquivo).
    """
    garantir_pasta(destino.parent)
    shutil.copy2(origem, destino)

# LEITURA E PREPARAÇÃO DOS DADOS
def carregar_dados_excel(caminho_excel: Path) -> pd.DataFrame:
    """
    Carrega a planilha e valida as colunas necessárias.

    Parâmetros
    ----------
    caminho_excel : Path
        Caminho do arquivo Excel.

    Retorno
    -------
    pd.DataFrame
        DataFrame com colunas padronizadas (LABEL, nome imagem, binários etc.)
    """
    df = pd.read_excel(caminho_excel)

    colunas_necessarias = [
        COL_LABEL,
        COL_NOME_IMG,
        COL_MODELO_ACERTOU,
        COL_GRADCAM_OK,
        COL_REANOTAR,
    ]

    ausentes = [c for c in colunas_necessarias if c not in df.columns]
    if ausentes:
        raise KeyError(f"Colunas obrigatórias ausentes na planilha: {ausentes}")

    # Padroniza LABEL
    df[COL_LABEL] = df[COL_LABEL].apply(normalizar_texto)

    # Padroniza nome da imagem (mantém como está, mas remove espaços nas pontas)
    df[COL_NOME_IMG] = df[COL_NOME_IMG].astype(str).str.strip()

    # Padroniza binários
    df[COL_MODELO_ACERTOU] = df[COL_MODELO_ACERTOU].apply(padronizar_binario)
    df[COL_GRADCAM_OK] = df[COL_GRADCAM_OK].apply(padronizar_binario)
    df[COL_REANOTAR] = df[COL_REANOTAR].apply(padronizar_binario)

    # Remove linhas sem informações mínimas
    df = df.dropna(subset=[COL_LABEL, COL_NOME_IMG, COL_MODELO_ACERTOU, COL_GRADCAM_OK]).copy()
    df = df[df[COL_LABEL].isin(["POSITIVO", "NEGATIVO"])].reset_index(drop=True)

    return df

# ESTRUTURA DE SAÍDA E REGRAS DE ROTEAMENTO
def nome_pasta_combinacao(modelo_acertou: int, gradcam_ok: int) -> str:
    """
    Gera o nome da pasta conforme a combinação de valores binários.

    Exemplo:
    - modelo_acertou=1, gradcam_ok=0 -> 'M1_G0'

    Retorno
    -------
    str
        Nome da pasta da combinação.
    """
    return f"Modelo{modelo_acertou}_GradCam{gradcam_ok}"


def montar_estrutura_saida(base_saida: Path) -> Dict[str, Dict[str, Path]]:
    """
    Cria a estrutura de pastas de saída conforme solicitado.

    Estrutura:
    base_saida/
      POSITIVO/
        M1_G0, M1_G1, M0_G1, M0_G0
      NEGATIVO/
        M1_G0, M1_G1, M0_G1, M0_G0
      REANOTACAO/
        POSITIVO/
        NEGATIVO/

    Retorno
    -------
    Dict[str, Dict[str, Path]]
        Mapa com caminhos criados para facilitar roteamento.
    """
    combos = [(1,0), (1,1), (0,1), (0,0)]
    labels = ["POSITIVO", "NEGATIVO"]

    saida_map: Dict[str, Dict[str, Path]] = {"POSITIVO": {}, "NEGATIVO": {}}

    # Pastas por label e combinação
    for label in labels:
        for m, g in combos:
            p = base_saida / label / nome_pasta_combinacao(m, g)
            garantir_pasta(p)
            saida_map[label][nome_pasta_combinacao(m, g)] = p

    # Pastas de reanotação
    reanot_base = base_saida / "REANOTACAO"
    garantir_pasta(reanot_base / "POSITIVO")
    garantir_pasta(reanot_base / "NEGATIVO")

    saida_map["REANOTACAO"] = {
        "POSITIVO": reanot_base / "POSITIVO",
        "NEGATIVO": reanot_base / "NEGATIVO"
    }

    return saida_map

# PROCESSAMENTO E CÓPIA DAS IMAGENS
@dataclass(frozen=True)
class ResultadoProcessamento:
    """
    Representa o resumo do processamento.
    """
    total_linhas: int
    copiados_classificacao: int
    copiados_reanotacao: int
    nao_encontrados: int
    registros_invalidos: int


def processar_copias(
    df: pd.DataFrame,
    pastas_origem: List[Path],
    saida_map: Dict[str, Dict[str, Path]],
    pasta_reanotacao_origem_neg: Path,
    pasta_reanotacao_origem_pos: Path
) -> ResultadoProcessamento:
    """
    Processa o DataFrame copiando imagens para:
    1) Pastas por LABEL e combinação (MODELO_ACERTOU x GRADCAM_OK)
       usando as pastas originais de classificação;
    2) Pasta de REANOTACAO/label quando REANOTAR = 1
       usando pastas específicas de reanotação.

    Parâmetros
    ----------
    df : pd.DataFrame
        Dados da planilha já preparados.
    pastas_origem : List[Path]
        Pastas onde as imagens estão para a organização principal
        (grad_negativo_parte1 e grad_positivo_parte1).
    saida_map : Dict[str, Dict[str, Path]]
        Mapeamento de pastas de saída.
    pasta_reanotacao_origem_neg : Path
        Pasta de origem das imagens NEGATIVAS usada apenas para reanotação.
    pasta_reanotacao_origem_pos : Path
        Pasta de origem das imagens POSITIVAS usada apenas para reanotação.

    Retorno
    -------
    ResultadoProcessamento
        Resumo do processamento.
    """
    total = len(df)
    copiados_cls = 0
    copiados_rea = 0
    nao_encontrados = 0
    invalidos = 0

    for _, row in df.iterrows():
        label = row[COL_LABEL]
        nome_img = row[COL_NOME_IMG]

        m = row[COL_MODELO_ACERTOU]
        g = row[COL_GRADCAM_OK]
        r = row.get(COL_REANOTAR, 0)

        # Valida binários essenciais
        if m not in (0, 1) or g not in (0, 1):
            invalidos += 1
            continue

        # ======================================================
        # 1) Cópia para pastas por combinação
        #    Usa grad_negativo_parte1 e grad_positivo_parte1
        # ======================================================
        origem_cls = localizar_imagem(nome_img, pastas_origem)
        if origem_cls is None:
            nao_encontrados += 1
            continue

        pasta_combo = saida_map[label][nome_pasta_combinacao(m, g)]
        destino_cls = pasta_combo / origem_cls.name
        copiar_arquivo(origem_cls, destino_cls)
        copiados_cls += 1

        # ======================================================
        # 2) Cópia para REANOTAÇÃO
        #    Usa pastas específicas hardcoded para reanotação
        # ======================================================
        if r == 1:
            origem_rea = localizar_imagem_reanotacao(
                nome_imagem=nome_img,
                label=label,
                pasta_reanotacao_neg=pasta_reanotacao_origem_neg,
                pasta_reanotacao_pos=pasta_reanotacao_origem_pos
            )

            if origem_rea is None:
                nao_encontrados += 1
            else:
                pasta_rea = saida_map["REANOTACAO"][normalizar_texto(label)]
                destino_rea = pasta_rea / origem_rea.name
                copiar_arquivo(origem_rea, destino_rea)
                copiados_rea += 1

    return ResultadoProcessamento(
        total_linhas=total,
        copiados_classificacao=copiados_cls,
        copiados_reanotacao=copiados_rea,
        nao_encontrados=nao_encontrados,
        registros_invalidos=invalidos
    )

def localizar_imagem_reanotacao(
    nome_imagem: str,
    label: str,
    pasta_reanotacao_neg: Path,
    pasta_reanotacao_pos: Path
) -> Optional[Path]:
    """
    Localiza a imagem nas pastas específicas usadas apenas para REANOTAÇÃO.

    Regras:
    - LABEL = 'NEGATIVO' -> procura em pasta_reanotacao_neg
    - LABEL = 'POSITIVO' -> procura em pasta_reanotacao_pos

    Parâmetros
    ----------
    nome_imagem : str
        Nome do arquivo da imagem.
    label : str
        Classe da imagem ('NEGATIVO' ou 'POSITIVO').
    pasta_reanotacao_neg : Path
        Pasta de origem das imagens NEGATIVAS para reanotação.
    pasta_reanotacao_pos : Path
        Pasta de origem das imagens POSITIVAS para reanotação.

    Retorno
    -------
    Optional[Path]
        Caminho da imagem se encontrada; caso contrário, None.
    """
    label_norm = normalizar_texto(label)

    if label_norm == "NEGATIVO":
        candidato = pasta_reanotacao_neg / nome_imagem
        return candidato if candidato.exists() and candidato.is_file() else None

    if label_norm == "POSITIVO":
        candidato = pasta_reanotacao_pos / nome_imagem
        return candidato if candidato.exists() and candidato.is_file() else None

    return None

In [ ]:
# LEITURA E PREPARAÇÃO DOS DADOS
# Valida entradas
validar_arquivos_e_pastas(
    ARQUIVO_EXCEL,
    PASTA_ORIGEM_NEG,
    PASTA_ORIGEM_POS,
    PASTA_REANOTACAO_ORIGEM_NEG,
    PASTA_REANOTACAO_ORIGEM_POS
)

df = carregar_dados_excel(ARQUIVO_EXCEL)

print(f"Total de registros válidos: {len(df)}")
df.head()

In [ ]:
# ESTRUTURA DE SAÍDA E REGRAS DE ROTEAMENTO
saida_map = montar_estrutura_saida(PASTA_SAIDA)
print("Estrutura de saída criada em:", PASTA_SAIDA.resolve())

In [ ]:
# PROCESSAMENTO E CÓPIA DAS IMAGENS
resultado = processar_copias(
    df=df,
    pastas_origem=[PASTA_ORIGEM_NEG, PASTA_ORIGEM_POS],
    saida_map=saida_map,
    pasta_reanotacao_origem_neg=PASTA_REANOTACAO_ORIGEM_NEG,
    pasta_reanotacao_origem_pos=PASTA_REANOTACAO_ORIGEM_POS
)

resultado

In [ ]:
# RELATÓRIO FINAL
print("=" * 80)
print("RELATÓRIO DE ORGANIZAÇÃO DE IMAGENS")
print("=" * 80)
print(f"Total de registros (válidos p/ processamento): {resultado.total_linhas}")
print(f"Imagens copiadas (pastas por combinação):      {resultado.copiados_classificacao}")
print(f"Imagens copiadas (REANOTACAO):                {resultado.copiados_reanotacao}")
print(f"Imagens NÃO encontradas nas pastas origem:    {resultado.nao_encontrados}")
print(f"Registros inválidos (binários ausentes/ruins): {resultado.registros_invalidos}")
print("\nSaída gerada em:", PASTA_SAIDA.resolve())
print("=" * 80)